# LLM-as-a-Judge Evaluation System for AIMO

This notebook uses an AI model as an **open-ended** judge to score the mathematical reasoning traces. Instead of relying on a simple majority vote or mathematical entropy to pick an answer, we ask a separate "judge" AI to read the step-by-step reasoning and give it a score based on a clear set of rules.

### How It Works:
- **Open-Ended Pattern:** The judging model reads the entire free-form reasoning text (the "trace") that the solver produced. It has no "golden answer key" or multiple choices. It must purely evaluate the math logic.
- **Rubric-based Scoring:** The judge grades each trace on three specific areas from 1 to 10:
  1. **Relevance:** Is the solver staying on topic and answering the actual problem?
  2. **Logical Correctness (Coherence):** Do the steps make mathematical sense? Do later equations naturally follow logically from earlier ones without contradicting?
  3. **Repetition & Hallucination:** Is the solver running around in circles repeating the same text, or making up facts without proof?
- **Code Score:** We do not ask the LLM to grade Python code because it usually gives inaccurate scores (like giving a high score even if the code crashes). Instead, we use a simple math formula to deduct points automatically based on the ratio of Python errors to Python calls.
- **Smart Grouping:** When multiple attempts arrive at the same answer, we group them together. First, traces scoring below a threshold are filtered out (unless all traces for an answer fail, in which case the best one is kept). We compute a *weighted average* of their rubrics (Relevance, Logic, Repetition, Code), and give them a small bonus if lots of traces vote for it (using the formula: `Average Score * log8(Vote Count + 1)`).

### References:
- [Exploring LLM-as-a-Judge (W&B)](https://wandb.ai/site/articles/exploring-llm-as-a-judge/)
- [<ins>Tutorial</ins>: Implementing LLM-as-a-Judge (W&B)](https://wandb.ai/byyoung3/judgebench/reports/Tutorial-Implementing-LLM-as-a-Judge-for-evaluation--VmlldzoxNTQ5OTk1OA)
- [Evidently AI Guide to LLM as a Judge](https://www.evidentlyai.com/llm-guide/llm-as-a-judge)
- [<ins>Paper</ins>: What Defines Good Reasoning in LLMs? Dissecting Reasoning Steps with Multi-Aspect Evaluation (arXiv:2510.20603v1)](https://arxiv.org/html/2510.20603v1)

In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires scikit-learn, which is not installed.
kauldron 1.3.0 requires tensorflow, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
stable-baselines3 2.1.0 requires matplotlib, which is not installed.
sentence-transformers 5.1.1 requires scikit-learn, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires matplotlib>=3.7.1, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
pynndescent 0.5.13 requires scikit-learn>=0.

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import json
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

### 1. Judge Configuration & Rubrics

In this section, we set up the prompt and rules for the LLM judge.
We tell the judge strictly to look at **Relevance**, **Logical Correctness**, and **Repetition**, and reply with a score between 1 and 10 in JSON format.

We also define some safety limits:
- **Skip the judge:** If multiple attempts quickly find the exact same answer without taking too much time, we just select that answer and skip the judge entirely to save inference budget.
- **Context Limits:** We set a max limit on how much text the judge can read so it doesn't crash the notebook.

In [9]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    # ── LLM-as-Judge Prompt ──────────────────────────────────────────────
    judge_prompt = (
        'You are an expert mathematical reasoning evaluator. Your task is to evaluate '
        'a single reasoning trace produced by a math solver attempting an IMO-level problem.\n\n'
        
        'You will evaluate the trace on exactly 3 rubrics, each scored 1-10.\n'
        'Code execution errors are scored separately and not your concern.\n\n'
        
        '# Rubric 1: Relevance (1-10)\n'
        'Does each reasoning step address the problem at hand? Is the work grounded '
        'in the problem statement and constraints?\n'
        '  1 = Completely off-topic or addresses a different problem\n'
        '  3 = Tangential portions but some steps are relevant\n'
        '  5 = Mostly relevant but contains unnecessary detours\n'
        '  7 = Nearly all steps directly address the problem\n'
        '  10 = Every step is tightly focused on solving the given problem\n\n'
        
        '# Rubric 2: Logical Correctness (1-10)\n'
        'Are the mathematical reasoning steps logically valid? Do later steps follow '
        'coherently from earlier ones? Does the solver correctly interpret code outputs '
        'and intermediate results? PENALIZE EXPONENTIALLY for:\n'
        '- Misinterpreting a computed value (e.g. taking a value at large N as the maximum '
        'when the actual max occurred at a smaller N)\n'
        '- Drawing wrong conclusions from correct computations\n'
        '- Off-by-one errors in counting arguments (e.g. ceil vs floor confusion)\n'
        '- Using the wrong mathematical framework entirely for the problem\n'
        '- Unjustified logical leaps that skip critical steps\n'
        '- Later steps contradicting or not following from earlier established results\n'
        '- Internal inconsistency where the reasoning changes direction without justification\n'
        '  1 = Critical logical errors that invalidate the entire reasoning chain\n'
        '  2 = Major logical error that fundamentally undermines the solution\n'
        '  4 = Significant logical flaw (e.g. wrong paradigm, misinterpreted result)\n'
        '  6 = Minor logical issue that does not undermine the overall argument\n'
        '  8 = Sound logic with trivial gaps that are easily filled\n'
        '  10 = Flawless logical reasoning throughout, every step follows from the previous\n\n'
        
        '# Rubric 3: Repetition & Hallucination (1-10)\n'
        'IMPORTANT DISTINCTIONS:\n'
        '- Trying DIFFERENT code approaches when one fails is NOT repetition (that is good debugging)\n'
        '- Repetition means: repeating the EXACT SAME or very similar reasoning text, '
        'or running identical/near-identical code blocks multiple times\n'
        '- Hallucination means: asserting mathematical facts without proof or justification, '
        'claiming a result was computed when it was not, or fabricating intermediate values\n'
        '  1 = Severe looping (same text/code repeated 3+ times) or major hallucinations\n'
        '  3 = Significant repetition of reasoning blocks or several fabricated claims\n'
        '  5 = Some repeated reasoning or minor hallucinations present\n'
        '  7 = Minimal repetition, nearly all claims backed by computation or proof\n'
        '  10 = No repetition, no hallucination, every claim is supported\n\n'
        
        '# OUTPUT FORMAT (MANDATORY):\n'
        'For each rubric, first write a SHORT rationale string, then the integer score.\n'
        'Evaluate each rubric independently. Do NOT let a score in one rubric influence the others.\n'
        'Output ONLY a valid JSON object with exactly these 6 keys:\n'
        '{\n'
        '  "relevance_rationale": "<1-2 sentences>", "relevance": <int 1-10>,\n'
        '  "logical_correctness_rationale": "<1-2 sentences identifying any logical errors>", "logical_correctness": <int 1-10>,\n'
        '  "repetition_rationale": "<1-2 sentences>", "repetition": <int 1-10>\n'
        '}\n\n'
        'Do NOT include any other text, markdown blocks, or surrounding wrappers. STRICTLY JSON.'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 270

    notebook_limit = 17600 if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 9800
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

    # ── LLM-as-Judge Parameters ──────────────────────────────────────────
    judge_early_stop_agreement = 4   # Skip judge if this many traces agree on an answer
    judge_min_time_threshold = 270   # Skip judge if problem solved under this many seconds
    judge_timeout = 30               # Per-trace judge timeout for submission mode (seconds)
    judge_context_tokens = 65536     # Max context for judge calls (same as solver context_tokens)
    judge_max_output_tokens = 1024   # Max output tokens for judge response (JSON only + short rationale)
    judge_temperature = 0.3          # Low temperature for consistent evaluation
    
    rubric_weights = [0.25, 0.3, 0.25, 0.2]  # [Relevance, Logic, Repetition, Code] sum = 1.0
    judge_score_threshold = 6.5      # Minimum trace score to be considered
    
    # Output directories (local validation only)
    judge_results_dir = '/kaggle/working/judge_results'
    reasoning_dir = '/kaggle/working/reasoning_traces'

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

### 2. A Simple Template for the Judge

While the main solver uses complex prompts and tools to run Python code, the judge just needs to read text. 
Here, we create a basic chat format (`AIMO3JudgeTemplate`) for the judge so it doesn't get confused trying to write code. We also tell it to put in less "thinking effort" (`ReasoningEffort.MEDIUM`).

In [12]:
class AIMO3JudgeTemplate:
    """
    Template for LLM-as-Judge evaluation calls.
    Uses MEDIUM reasoning effort and NO tools.
    """

    def __init__(self):
        pass

    def get_system_content(self, judge_prompt: str) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(judge_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.MEDIUM)
        )

    def apply_chat_template(
        self, 
        judge_prompt: str, 
        user_prompt: str
    ) -> list[Message]:

        system_content = self.get_system_content(judge_prompt)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [13]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [14]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

### 3. The Main Solver and Judge Logic

This is where everything comes together. The `AIMO3Solver` class generates the reasoning traces and then grades them using the rules we set up above.

**Key Steps Inside:**
1. **`_compute_code_score`:** This score decays exponentially w.r.t the ratio of Python errors and Python calls. So, many errors in a few calls will be penalized more if a few errors are made among many calls. It uses the function `10 * exp(-2 * error_ratio)`.
2. **`_judge_single_trace`:** Sends the solver's text to the LLM judge to be graded. If the text is way too long, it dynamically cuts off parts of the middle/end so the judge doesn't crash from reading too much.
3. **`_should_run_judge`:** Checks if we even need to use the judge. (We skip it if the solver was fast and many answers agree).
4. **`_select_answer_with_judge`:** This combines the scores to pick a final winner. First, traces with an average score below the threshold (`judge_score_threshold`) are thrown out. However, if all traces in an answer group are below the threshold, the highest scoring one is kept so the group is not completely empty. Then, it groups matching answers, calculates the weighted average of the rubrics (`rubric_weights`), and adds a bonus for how frequently it was answered using: `Average Score * log8(Vote Count + 1)`. The answer with the highest final weight is chosen.

In [15]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.judge_template = AIMO3JudgeTemplate()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        # Create output directories (only for local validation)
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            os.makedirs(self.cfg.judge_results_dir, exist_ok=True)
            os.makedirs(self.cfg.reasoning_dir, exist_ok=True)

        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50 if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 28
        self.problem_counter = 0
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'FullReasoning': '[SKIPPED] Attempt was skipped (stop_event set or deadline passed).'
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        # Accumulate full reasoning trace across all turns
        reasoning_parts = []
        turn_number = 0
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    reasoning_parts.append('\n--- [STOPPED: early stop or deadline reached] ---\n')
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    reasoning_parts.append('\n--- [STOPPED: context window exhausted] ---\n')
                    break

                turn_number += 1
    
                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0, deadline - time.time()),
                    )
                except Exception as e:
                    reasoning_parts.append(f'\n--- [ERROR: Failed to create stream: {e}] ---\n')
                    break

                if stream is None:
                    continue
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                # Capture model output text for this turn
                turn_text = ''.join(text_chunks)
                reasoning_parts.append(f'[Turn {turn_number} - Assistant]\n{turn_text}\n')

                if final_answer is not None:
                    break
    
                if not token_buffer:
                    reasoning_parts.append('\n--- [STOPPED: empty token buffer] ---\n')
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    python_code = last_message.content[0].text
                    reasoning_parts.append(f'[Turn {turn_number} - Python Code]\n{python_code}\n')

                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
                    reasoning_parts.append(f'[Turn {turn_number} - Python Output]\n{response_text}\n')
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
            reasoning_parts.append(f'\n--- [EXCEPTION: {exc}] ---\n')
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        full_reasoning = '\n'.join(reasoning_parts)
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer,
            'FullReasoning': full_reasoning
        }

    # ── LLM-as-Judge Methods ─────────────────────────────────────────────

    def _build_judge_user_prompt(self, problem: str, trace: str) -> str:
        """
        Build the user-facing prompt for the judge.
        Contains the math problem and the full reasoning trace to evaluate.
        """
        # Truncate trace if too long to fit in judge context
        # Reserve tokens for system prompt + output
        max_trace_chars = self.cfg.judge_context_tokens * 3  # ~3 chars per token estimate
        if len(trace) > max_trace_chars:
            trace = trace[:max_trace_chars] + '\n\n[... TRACE TRUNCATED DUE TO LENGTH ...]'
        
        return (
            '# MATH PROBLEM:\n'
            f'{problem}\n\n'
            '# REASONING TRACE TO EVALUATE:\n'
            f'{trace}\n\n'
            '# TASK:\n'
            'Evaluate the above reasoning trace on the 3 rubrics (Relevance, Logical Correctness, '
            'Repetition & Hallucination). Provide rationales and output ONLY the JSON object.'
        )

    def _parse_judge_response(self, response_text: str) -> dict:
        """
        Parse the judge's JSON response. Returns 3-rubric scores dict.
        Falls back to default scores on parse failure.
        """
        default_scores = {
            'relevance_rationale': '', 'relevance': 7.5, 
            'logical_correctness_rationale': '', 'logical_correctness': 7.5, 
            'repetition_rationale': '', 'repetition': 7.5
        }
        
        score_keys = ['relevance', 'logical_correctness', 'repetition']
        
        if not response_text or not response_text.strip():
            return default_scores
        
        # Try to extract JSON from the response
        text = response_text.strip()
        
        def _extract(scores_dict):
            result = {}
            for k in default_scores:
                if k in score_keys:
                    result[k] = max(1, min(10, int(scores_dict[k])))
                else:
                    result[k] = str(scores_dict.get(k, ''))
            return result
        
        # Try direct parse first
        try:
            scores = json.loads(text)
            if isinstance(scores, dict) and all(k in scores for k in score_keys):
                return _extract(scores)
        except (json.JSONDecodeError, ValueError, TypeError):
            pass
        
        # Try to find JSON object in the text (greedy to capture nested strings)
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            try:
                scores = json.loads(json_match.group())
                if isinstance(scores, dict) and all(k in scores for k in score_keys):
                    return _extract(scores)
            except (json.JSONDecodeError, ValueError, TypeError):
                pass
        
        # Last resort: try regex extraction for individual scores  
        parsed = {}
        for key in score_keys:
            match = re.search(rf'"{key}"\s*:\s*(\d+)', text)
            if match:
                parsed[key] = max(1, min(10, int(match.group(1))))
        
        if len(parsed) == len(score_keys):
            parsed['relevance_rationale'] = ''
            parsed['logical_correctness_rationale'] = ''
            parsed['repetition_rationale'] = ''
            return parsed
        
        print(f'⚠️ Judge parse failure, using defaults. Response: {text[:200]}')
        return default_scores

    @staticmethod
    def _compute_code_score(python_calls: int, python_errors: int) -> float:
        """
        Compute code correctness score deterministically from error ratio.
        Uses exponential decay: score = 10 * exp(-2 * error_ratio)
        
        Examples:
          0 errors / 10 calls -> 10.0
          1 error  / 10 calls -> 8.2
          2 errors / 10 calls -> 6.7
          3 errors / 10 calls -> 5.5
          5 errors / 10 calls -> 3.7
          6 errors / 32 calls -> 6.9
          10 errors / 10 calls -> 1.4 -> clamped to 1
          0 calls (no code)   -> 8.0 (neutral)
        """
        if python_calls == 0:
            return 8.0  # No code used; neutral score
        
        error_ratio = python_errors / python_calls
        raw = 10.0 * math.exp(-2.0 * error_ratio)
        return round(max(1.0, min(10.0, raw)), 2)

    def _judge_single_trace(self, problem: str, result: dict, deadline: float) -> dict:
        """
        Judge a single reasoning trace. Returns result dict enriched with rubric scores.
        Code correctness is computed deterministically from python_errors/python_calls.
        The LLM judge evaluates: relevance, logical_correctness, repetition.
        """
        trace = result.get('FullReasoning', '')
        attempt = result['Attempt']
        answer = result['Answer']
        python_calls = result.get('Python Calls', 0)
        python_errors = result.get('Python Errors', 0)
        
        # Compute code score deterministically (never trust the LLM for this)
        code_score = self._compute_code_score(python_calls, python_errors)
        
        # Default scores for LLM-judged rubrics
        default_llm_scores = {
            'relevance_rationale': '', 'relevance': 7.5, 
            'logical_correctness_rationale': '', 'logical_correctness': 7.5, 
            'repetition_rationale': '', 'repetition': 7.5
        }
        
        if not trace or trace.startswith('[SKIPPED]') or time.time() > deadline:
            w = self.cfg.rubric_weights
            total = 7.5 * w[0] + 7.5 * w[1] + 7.5 * w[2] + code_score * w[3]
            return {
                **result,
                **default_llm_scores,
                'code_correctness': code_score,
                'JudgeScore': round(total, 4),
                'JudgeStatus': 'skipped'
            }
        
        try:
            # Build judge conversation
            user_prompt = self._build_judge_user_prompt(problem, trace)
            messages = self.judge_template.apply_chat_template(
                self.cfg.judge_prompt, 
                user_prompt
            )
            
            conversation = Conversation.from_messages(messages)
            prompt_ids = self.encoding.render_conversation_for_completion(
                conversation, Role.ASSISTANT
            )
            
            max_tokens = min(
                self.cfg.judge_max_output_tokens, 
                self.cfg.judge_context_tokens - len(prompt_ids)
            )
            
            # If trace overflows context, truncate and rebuild
            if max_tokens < 64:
                tokens_over = len(prompt_ids) - (self.cfg.judge_context_tokens - self.cfg.judge_max_output_tokens)
                chars_to_remove = (tokens_over + 256) * 4  # +256 token buffer, ~4 chars/token
                new_trace_len = max(200, len(trace) - chars_to_remove)
                
                if new_trace_len < 200:
                    print(f'⚠️ Attempt {attempt}: Judge context too full even for minimal trace, skipping.')
                    w = self.cfg.rubric_weights
                    total = 7.5 * w[0] + 7.5 * w[1] + 7.5 * w[2] + code_score * w[3]
                    return {
                        **result, **default_llm_scores,
                        'code_correctness': code_score,
                        'JudgeScore': round(total, 4), 'JudgeStatus': 'context_overflow'
                    }
                
                truncated_trace = trace[:new_trace_len] + '\n\n[... TRACE TRUNCATED TO FIT JUDGE CONTEXT ...]'
                print(f'ℹ️ Attempt {attempt}: Truncating trace from {len(trace)} to {new_trace_len} chars to fit judge context.')
                
                user_prompt = self._build_judge_user_prompt(problem, truncated_trace)
                messages = self.judge_template.apply_chat_template(
                    self.cfg.judge_prompt, user_prompt
                )
                conversation = Conversation.from_messages(messages)
                prompt_ids = self.encoding.render_conversation_for_completion(
                    conversation, Role.ASSISTANT
                )
                max_tokens = min(
                    self.cfg.judge_max_output_tokens,
                    self.cfg.judge_context_tokens - len(prompt_ids)
                )
                
                if max_tokens < 64:
                    print(f'⚠️ Attempt {attempt}: Still too full after truncation, skipping.')
                    w = self.cfg.rubric_weights
                    total = 7.5 * w[0] + 7.5 * w[1] + 7.5 * w[2] + code_score * w[3]
                    return {
                        **result, **default_llm_scores,
                        'code_correctness': code_score,
                        'JudgeScore': round(total, 4), 'JudgeStatus': 'context_overflow'
                    }
            
            # Determine timeout
            if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
                judge_timeout = min(self.cfg.judge_timeout, max(5, deadline - time.time()))
            else:
                judge_timeout = max(5, deadline - time.time())
            
            # Non-streaming call for judge (short output)
            response = self.client.completions.create(
                model=self.cfg.served_model_name,
                temperature=self.cfg.judge_temperature,
                max_tokens=max_tokens,
                prompt=prompt_ids,
                seed=self.cfg.seed,
                stream=False,
                extra_body={
                    'min_p': 0.01,
                    'stop_token_ids': self.stop_token_ids,
                },
                timeout=judge_timeout,
            )
            
            response_text = response.choices[0].text if response.choices else ''
            llm_scores = self._parse_judge_response(response_text)
            
            # Combine: 3 LLM-judged rubrics + 1 computed code score
            rel = llm_scores.get('relevance', 7.5)
            logic = llm_scores.get('logical_correctness', 7.5)
            rep = llm_scores.get('repetition', 7.5)
            w = self.cfg.rubric_weights
            total = rel * w[0] + logic * w[1] + rep * w[2] + code_score * w[3]
            
            return {
                **result,
                **llm_scores,
                'code_correctness': code_score,
                'JudgeScore': round(total, 4),
                'JudgeStatus': 'ok'
            }
            
        except Exception as exc:
            print(f'⚠️ Attempt {attempt}: Judge failed: {exc}')
            w = self.cfg.rubric_weights
            total = 7.5 * w[0] + 7.5 * w[1] + 7.5 * w[2] + code_score * w[3]
            return {
                **result, **default_llm_scores,
                'code_correctness': code_score,
                'JudgeScore': round(total, 4), 'JudgeStatus': f'error: {exc}'
            }

    def _should_run_judge(self, detailed_results: list, used_time: float) -> bool:
        """
        Determine if the judge should be invoked.
        
        Judge is SKIPPED (majority vote used instead) when BOTH conditions are met:
          1. Problem was solved in under judge_min_time_threshold seconds
          2. At least judge_early_stop_agreement traces agree on the same answer
        
        If EITHER condition is NOT met, the judge runs.
        """
        valid_answers = [r['Answer'] for r in detailed_results if r['Answer'] is not None]
        
        if not valid_answers:
            return False  # No valid answers, nothing to judge
        
        counts = Counter(valid_answers).most_common(1)
        most_common_count = counts[0][1] if counts else 0
        
        fast_enough = used_time < self.cfg.judge_min_time_threshold
        strong_agreement = most_common_count >= self.cfg.judge_early_stop_agreement
        
        if fast_enough and strong_agreement:
            print(f'⏭️ Skipping judge: solved in {used_time:.1f}s < {self.cfg.judge_min_time_threshold}s '
                  f'AND {most_common_count} traces agree (>= {self.cfg.judge_early_stop_agreement})')
            return False
        
        reasons = []
        if not fast_enough:
            reasons.append(f'time {used_time:.1f}s >= {self.cfg.judge_min_time_threshold}s')
        if not strong_agreement:
            reasons.append(f'max agreement {most_common_count} < {self.cfg.judge_early_stop_agreement}')
        
        print(f'⚖️ Running judge: {" AND ".join(reasons)}')
        return True

    def _run_judge(self, detailed_results: list, problem: str, deadline: float) -> list:
        """
        Run the LLM-as-Judge on all traces with valid answers, in parallel.
        Returns a list of result dicts enriched with judge scores.
        """
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        
        if not valid_results:
            return []
        
        judge_start = time.time()
        print(f'\n🔍 Judging {len(valid_results)} traces...')
        
        judged_results = []
        
        with ThreadPoolExecutor(max_workers=min(len(valid_results), self.cfg.workers)) as executor:
            futures = {
                executor.submit(
                    self._judge_single_trace, problem, result, deadline
                ): result['Attempt'] 
                for result in valid_results
            }
            
            for future in as_completed(futures):
                try:
                    judged = future.result()
                    judged_results.append(judged)
                except Exception as exc:
                    attempt_num = futures[future]
                    print(f'⚠️ Judge future failed for attempt {attempt_num}: {exc}')
        
        judge_elapsed = time.time() - judge_start
        print(f'⚖️ Judge completed in {judge_elapsed:.2f}s\n')
        
        return judged_results

    def _select_answer(self, detailed_results: list, judged_results: list | None) -> int:
        """
        Select the final answer.
        
        Two paths:
        1. Without judge (judged_results is None): Simple majority vote.  
        2. With judge: Group by answer -> avg rubric score per group ->
           group_weight = avg_score * log_base8(count + 1) -> pick highest.
        """
        if judged_results is not None and len(judged_results) > 0:
            return self._select_answer_with_judge(judged_results)
        else:
            return self._select_answer_majority(detailed_results)

    def _select_answer_majority(self, detailed_results: list) -> int:
        """Simple majority vote. Tiebreak by first occurrence."""
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        
        if not valid_results:
            print('\nNo valid answers found.')
            return 0
        
        answer_counts = Counter(r['Answer'] for r in valid_results)
        
        vote_data = []
        for answer, count in answer_counts.most_common():
            vote_data.append((answer, count))
        
        vote_df = pd.DataFrame(vote_data, columns=['Answer', 'Votes'])
        display(vote_df)
        
        final_answer = answer_counts.most_common(1)[0][0]
        print(f'\n[Majority Vote] Final Answer: {final_answer}\n')
        
        return final_answer

    def _select_answer_with_judge(self, judged_results: list) -> int:
        """
        Weighted answer selection using judge scores.
        group_weight = avg_score * log_base8(count + 1)
        """
        # Group by answer
        answer_groups = defaultdict(list)
        for r in judged_results:
            if r['Answer'] is not None:
                answer_groups[r['Answer']].append(r)
        
        if not answer_groups:
            print('\nNo valid judged answers.')
            return 0
        
        log_base = 8.0
        scored_answers = []
        
        for answer, group in answer_groups.items():
            valid_traces = [r for r in group if r['JudgeScore'] >= self.cfg.judge_score_threshold]
            
            if not valid_traces:
                # If all are rejected, keep the one with the highest score
                best_trace = max(group, key=lambda x: x['JudgeScore'])
                valid_traces = [best_trace]
                
            scores = [r['JudgeScore'] for r in valid_traces]
            avg_score = sum(scores) / len(scores)
            count = len(valid_traces)
            group_weight = avg_score * (math.log(count + 1) / math.log(log_base))
            
            # Per-rubric averages for display
            avg_rel = sum(r.get('relevance', 7.5) for r in valid_traces) / count
            avg_logic = sum(r.get('logical_correctness', 7.5) for r in valid_traces) / count
            avg_code = sum(r.get('code_correctness', 8.0) for r in valid_traces) / count
            avg_rep = sum(r.get('repetition', 7.5) for r in valid_traces) / count
            
            scored_answers.append({
                'Answer': answer,
                'Votes': count,
                'Avg Score': round(avg_score, 3),
                'Weight': round(group_weight, 3),
                'Relevance': round(avg_rel, 1),
                'Logic': round(avg_logic, 1),
                'Code': round(avg_code, 1),
                'Repetition': round(avg_rep, 1),
            })
        
        scored_answers.sort(key=lambda x: x['Weight'], reverse=True)
        
        vote_df = pd.DataFrame(scored_answers)
        display(vote_df)
        
        final_answer = scored_answers[0]['Answer']
        final_weight = scored_answers[0]['Weight']
        final_votes = scored_answers[0]['Votes']
        
        print(f'\n[Judge Weighted] Final Answer: {final_answer} | '
              f'Votes: {final_votes} | Weight: {final_weight:.3f}\n')
        
        return final_answer

    def _save_judge_results_csv(
        self, 
        judged_results: list, 
        ground_truth: int | None,
        problem_id: int,
        problem_text: str
    ) -> str | None:
        """Save judge evaluation results to CSV for analysis (local validation only)."""
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None
        
        if not judged_results:
            return None
        
        rows = []
        for r in sorted(judged_results, key=lambda x: x['Attempt']):
            answer = r['Answer']
            if ground_truth is not None and answer is not None:
                verdict = 'correct' if answer == ground_truth else 'wrong'
            elif answer is None:
                verdict = 'unfinished'
            else:
                verdict = 'unknown'
            
            rows.append({
                'problem_id': problem_id,
                'attempt': r['Attempt'],
                'answer': answer if answer is not None else '',
                'ground_truth': ground_truth if ground_truth is not None else '',
                'verdict': verdict,
                'total_tokens': r['Response Length'],
                'python_calls': r['Python Calls'],
                'python_errors': r['Python Errors'],
                'relevance': r.get('relevance', ''),
                'logical_correctness': r.get('logical_correctness', ''),
                'code_correctness': r.get('code_correctness', ''),
                'repetition': r.get('repetition', ''),
                'judge_score': r.get('JudgeScore', ''),
                'judge_status': r.get('JudgeStatus', ''),
                'relevance_rationale': r.get('relevance_rationale', ''),
                'logical_correctness_rationale': r.get('logical_correctness_rationale', ''),
                'repetition_rationale': r.get('repetition_rationale', '')
            })
        
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.cfg.judge_results_dir, f'problem_{problem_id}_judge.csv')
        df.to_csv(csv_path, index=False)
        
        verdicts = df['verdict'].value_counts().to_dict()
        verdict_str = ', '.join(f'{v}: {c}' for v, c in verdicts.items())
        print(f'📝 Judge results saved: {csv_path} ({len(rows)} attempts | {verdict_str})')
        return csv_path

    def _save_reasoning_csv(
        self,
        detailed_results: list,
        ground_truth: int | None,
        problem_id: int,
        problem_text: str
    ) -> str | None:
        """Save full reasoning traces to CSV (local validation only)."""
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None
        
        rows = []
        for r in sorted(detailed_results, key=lambda x: x['Attempt']):
            answer = r['Answer']
            if answer is None:
                verdict = 'unfinished'
            elif ground_truth is not None:
                verdict = 'correct' if answer == ground_truth else 'wrong'
            else:
                verdict = 'unknown'
            
            rows.append({
                'problem_id': problem_id,
                'problem_text': problem_text[:500],
                'attempt': r['Attempt'],
                'answer': answer if answer is not None else '',
                'ground_truth': ground_truth if ground_truth is not None else '',
                'verdict': verdict,
                'total_tokens': r['Response Length'],
                'python_calls': r['Python Calls'],
                'python_errors': r['Python Errors'],
                'full_reasoning': r.get('FullReasoning', '')
            })
        
        if not rows:
            return None
        
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.cfg.reasoning_dir, f'problem_{problem_id}_reasoning.csv')
        df.to_csv(csv_path, index=False)
        
        verdicts = df['verdict'].value_counts().to_dict()
        verdict_str = ', '.join(f'{v}: {c}' for v, c in verdicts.items())
        print(f'📝 Reasoning traces saved: {csv_path} ({len(rows)} attempts | {verdict_str})')
        return csv_path

    def solve_problem(self, problem: str, ground_truth_answer: int | None = None) -> int:
    
        problem_start_time = time.time()
        self.problem_counter += 1
        problem_id = self.problem_counter
        
        print(f'\nProblem {problem_id}: {problem[:200]}...\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f}s | Problems remaining: {self.problems_remaining}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
    
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
    
                        for f in futures:
                            f.cancel()
    
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        # Calculate time used for inference
        used_time = time.time() - problem_start_time
        
        if detailed_results:
            # Display attempt results table
            display_results = []
            for r in detailed_results:
                display_results.append({
                    'Attempt': r['Attempt'],
                    'Answer': r['Answer'],
                    'Response Length': r['Response Length'],
                    'Python Calls': r['Python Calls'],
                    'Python Errors': r['Python Errors']
                })
            
            results_df = pd.DataFrame(display_results)
            results_df['Answer'] = results_df['Answer'].astype('Int64')
            display(results_df)
    
        if not valid_answers:
            print(f'\n[Inference] Took {used_time:.2f}s | No valid answers found')
            print('\nResult: 0\n')
            self._save_reasoning_csv(detailed_results, ground_truth_answer, problem_id, problem)
            return 0

        # ── Judge Decision ───────────────────────────────────────────────
        judged_results = None
        
        if self._should_run_judge(detailed_results, used_time):
            # Recalculate deadline: use remaining budget for judge
            judge_deadline = time.time() + max(60, deadline - time.time())
            judged_results = self._run_judge(detailed_results, problem, judge_deadline)
            
            # Display judge scores
            if judged_results:
                judge_display = []
                for r in sorted(judged_results, key=lambda x: x['Attempt']):
                    judge_display.append({
                        'Attempt': r['Attempt'],
                        'Answer': r['Answer'],
                        'Relevance': r.get('relevance', ''),
                        'Logic': r.get('logical_correctness', ''),
                        'Code': r.get('code_correctness', ''),
                        'Repetition': r.get('repetition', ''),
                        'JudgeScore': r.get('JudgeScore', ''),
                        'Status': r.get('JudgeStatus', ''),
                    })
                judge_df = pd.DataFrame(judge_display)
                judge_df['Answer'] = judge_df['Answer'].astype('Int64')
                display(judge_df)

        final_answer = self._select_answer(detailed_results, judged_results)
        
        print(f'[Inference] Took {used_time:.2f}s | Budget was {budget:.2f}s\n')
        
        # Save results for analysis (local validation only)
        self._save_reasoning_csv(detailed_results, ground_truth_answer, problem_id, problem)
        if judged_results:
            self._save_judge_results_csv(judged_results, ground_truth_answer, problem_id, problem)

        return final_answer
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [16]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 71.02 seconds.

Waiting for vLLM server...
Server is ready (took 120.54 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.22 seconds.



In [17]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/datasets/nahidhossainredom/omni-math-hardestdifficulty-9/omni_math_hard.csv"
)

# df = df[df['id']==12].copy()

df = df[['id','problem','answer']]

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

print(f"Dataset prepared with {len(df)} problems.")

Dataset prepared with 28 problems.


In [18]:
# # Load reference data and keep ground truth for accuracy calculation
# df = pd.read_csv(
#     "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"
# )

# # Store ground truth answers for accuracy calculation (only in local mode)
# ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# # Create input file without answers
# df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# # Track predictions for accuracy calculation
# predictions = {}
# correct_count = 0
# total_count = 0

In [19]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    # Get ground truth for analysis (only available in local validation)
    gt_answer = ground_truth.get(question_id, None)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text, ground_truth_answer=gt_answer)
    predictions[question_id] = final_answer
    
    gc.enable()
    gc.collect()

    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [20]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))

------
ID: 1762
Question: Lucy starts by writing $s$ integer-valued $2022$-tuples on a blackboard. After doing that, she can take any two (not necessarily distinct) tuples $\mathbf{v}=(v_1,\ldots,v_{2022})$ and $\mathbf{w}=(w_...

Problem 1: Lucy starts by writing $s$ integer-valued $2022$-tuples on a blackboard. After doing that, she can take any two (not necessarily distinct) tuples $\mathbf{v}=(v_1,\ldots,v_{2022})$ and $\mathbf{w}=(w_...

Budget: 900.00s | Problems remaining: 28



,Attempt,Answer,Response Length,Python Calls,Python Errors
0,6,2022,25583,18,4
1,4,2023,29377,13,1
2,2,2022,34715,35,7
3,1,12,35857,25,11
4,7,2022,41129,15,4
5,3,2022,39962,29,7


⚖️ Running judge: time 455.0s >= 270s

🔍 Judging 6 traces...
⚠️ Judge parse failure, using defaults. Response: analysisWe need to evaluate the reasoning trace (the assistant's answer) for the problem. The trace includes a long reasoning, constructing a solution that minimal s = 2022, with arguments for lower b
⚖️ Judge completed in 22.77s



,Attempt,Answer,Relevance,Logic,Code,Repetition,JudgeScore,Status
0,1,12,8.0,6.0,4.15,7.0,6.380,ok
1,2,2022,7.5,7.5,6.70,7.5,7.340,ok
2,3,2022,10.0,9.0,6.17,10.0,8.934,ok
3,4,2023,10.0,10.0,8.57,10.0,9.714,ok
4,6,2022,10.0,9.0,6.41,10.0,8.982,ok
5,7,2022,10.0,9.0,5.87,10.0,8.874,ok


,Answer,Votes,Avg Score,Weight,Relevance,Logic,Code,Repetition
0,2022,4,8.532,6.604,9.4,8.6,6.3,9.4
1,2023,1,9.714,3.238,10.0,10.0,8.6,10.0
2,12,1,6.380,2.127,8.0,6.0,4.2,7.0


...


[Judge Weighted] Final Answer: 1 | Votes: 4 | Weight: 6.434

[Inference] Took 487.62s | Budget was 900.00s

📝 Reasoning traces saved: /kaggle/working/reasoning_traces/problem_27_reasoning.csv (4 attempts | correct: 4)
📝 Judge results saved: /kaggle/working/judge_results/problem_27_judge.csv (4 attempts | correct: 4)
Answer: 1 | Ground Truth: 1 | ✅
📊 Running Accuracy: 20/27 (74.1%)
------

------
ID: 1767
Question: Players $A$ and $B$ play a game on a blackboard that initially contains 2020 copies of the number 1 . In every round, player $A$ erases two numbers $x$ and $y$ from the blackboard, and then player $B$...

Problem 28: Players $A$ and $B$ play a game on a blackboard that initially contains 2020 copies of the number 1 . In every round, player $A$ erases two numbers $x$ and $y$ from the blackboard, and then player $B$...

Budget: 900.00s | Problems remaining: 1



,Attempt,Answer,Response Length,Python Calls,Python Errors
0,7,7,15824,5,1
1,8,7,18306,10,0
2,2,7,19351,8,1
3,3,7,23880,22,2


⏭️ Skipping judge: solved in 245.7s < 270s AND 4 traces agree (>= 4)


,Answer,Votes
0,7,4



[Majority Vote] Final Answer: 7

[Inference] Took 245.71s | Budget was 900.00s

📝 Reasoning traces saved: /kaggle/working/reasoning_traces/problem_28_reasoning.csv (4 attempts | correct: 4)
Answer: 7 | Ground Truth: 7 | ✅
📊 Running Accuracy: 21/28 (75.0%)
------

